In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

from statsmodels.tsa.arima.model import ARIMA

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [3]:
df = pd.read_csv(r"D:\Projects\ML PROJECT 2\dataset\election_data.csv")

In [4]:
df.head()

,st_name,year,pc_no,pc_name,pc_type,cand_name,cand_sex,partyname,partyabbre,totvotpoll,electors
0,Andaman & Nicobar Islands,1977,1,Andaman & Nicobar Islands,GEN,K.R. Ganesh,M,Independents,IND,25168,85308
1,Andaman & Nicobar Islands,1977,1,Andaman & Nicobar Islands,GEN,Manoranjan Bhakta,M,Indian National Congress,INC,35400,85308
2,Andaman & Nicobar Islands,1980,1,Andaman & Nicobar Islands,GEN,Ramesh Mazumdar,M,Independents,IND,109,96084
3,Andaman & Nicobar Islands,1980,1,Andaman & Nicobar Islands,GEN,Alagiri Swamy,M,Independents,IND,125,96084
4,Andaman & Nicobar Islands,1980,1,Andaman & Nicobar Islands,GEN,Kannu Chemy,M,Independents,IND,405,96084


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73081 entries, 0 to 73080
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   st_name     73081 non-null  str  
 1   year        73081 non-null  int64
 2   pc_no       73081 non-null  int64
 3   pc_name     73081 non-null  str  
 4   pc_type     65011 non-null  str  
 5   cand_name   73081 non-null  str  
 6   cand_sex    72539 non-null  str  
 7   partyname   73081 non-null  str  
 8   partyabbre  73081 non-null  str  
 9   totvotpoll  73081 non-null  int64
 10  electors    73081 non-null  int64
dtypes: int64(4), str(7)
memory usage: 6.1 MB


In [6]:
df.describe()

,year,pc_no,totvotpoll,electors
count,73081.000000,73081.000000,73081.000000,7.308100e+04
mean,1996.779505,22.311490,49834.760266,1.122277e+06
std,10.432527,19.039793,104893.319706,3.560049e+05
min,1977.000000,1.000000,0.000000,1.947100e+04
25%,1989.000000,7.000000,872.000000,9.129850e+05
50%,1996.000000,18.000000,2743.000000,1.099503e+06
75%,2004.000000,33.000000,19185.000000,1.329086e+06
max,2014.000000,85.000000,863358.000000,3.368399e+06


In [7]:
df.isnull().sum()

st_name          0
year             0
pc_no            0
pc_name          0
pc_type       8070
cand_name        0
cand_sex       542
partyname        0
partyabbre       0
totvotpoll       0
electors         0
dtype: int64

In [10]:
# Fill numeric columns with 0
num_cols = df.select_dtypes(include=['number']).columns
df[num_cols] = df[num_cols].fillna(0)

# Fill text columns with 'Unknown'
str_cols = df.select_dtypes(include=['object', 'string']).columns
df[str_cols] = df[str_cols].fillna('Unknown')

In [12]:
print(df.columns.tolist())

['st_name', 'year', 'pc_no', 'pc_name', 'pc_type', 'cand_name', 'cand_sex', 'partyname', 'partyabbre', 'totvotpoll', 'electors']


In [15]:
print(df.shape)

(73081, 12)


In [16]:
print(df['winner'].value_counts())

winner
0    72162
1      919
Name: count, dtype: int64


In [17]:
df['winner'] = (
    df.groupby(['year', 'pc_no'])['totvotpoll']
      .transform('max')
      .eq(df['totvotpoll'])
      .astype(int)
)

In [18]:
print(df[['year','pc_name','cand_name','totvotpoll','winner']].head(20))

    year                    pc_name           cand_name  totvotpoll  winner
0   1977  Andaman & Nicobar Islands         K.R. Ganesh       25168       0
1   1977  Andaman & Nicobar Islands   Manoranjan Bhakta       35400       0
2   1980  Andaman & Nicobar Islands     Ramesh Mazumdar         109       0
3   1980  Andaman & Nicobar Islands       Alagiri Swamy         125       0
4   1980  Andaman & Nicobar Islands         Kannu Chemy         405       0
5   1980  Andaman & Nicobar Islands           K.N. Raju         470       0
6   1980  Andaman & Nicobar Islands  Rajender Lall Saha         717       0
7   1980  Andaman & Nicobar Islands         Karpu Swamy        1123       0
8   1980  Andaman & Nicobar Islands     Samar Choudhury        2034       0
9   1980  Andaman & Nicobar Islands      K. Kanda Swamy       15856       0
10  1980  Andaman & Nicobar Islands       P.K.S. Prasad       16014       0
11  1980  Andaman & Nicobar Islands   Manoranjan Bhakta       42046       0
12  1984  An

In [19]:
df[df['winner'] == 1].head(10)

,st_name,year,pc_no,pc_name,pc_type,cand_name,cand_sex,partyname,partyabbre,totvotpoll,electors,winner
138,Andhra Pradesh,1977,11,Eluru,GEN,Kommareddy Suryanarayana,M,Indian National Congress,INC,290410,669778,1
244,Andhra Pradesh,1977,39,Warangal,GEN,S. B. Giri,M,Indian National Congress,INC,251211,581239,1
329,Andhra Pradesh,1980,11,Eluru,GEN,Chittoori Subbarao Chowdary,M,Indian Natioanl Congress (I),INC(I),266805,732089,1
377,Andhra Pradesh,1980,19,Nellore,SC,D. Kamakshaiah,M,Indian Natioanl Congress (I),INC(I),294326,816504,1
402,Andhra Pradesh,1980,23,Cuddapah,GEN,K. Obul Reddy,M,Indian Natioanl Congress (I),INC(I),256204,774582,1
501,Andhra Pradesh,1980,38,Hanamkonda,GEN,P. V. Narsimha Rao,M,Indian Natioanl Congress (I),INC(I),257961,692790,1
562,Andhra Pradesh,1984,6,Anakapalli,GEN,Appalanarasimham P.,M,Telugu Desam,TDP,322347,744527,1
574,Andhra Pradesh,1984,8,Rajahmundry,GEN,Srihari Rao,M,Telugu Desam,TDP,381091,815302,1
584,Andhra Pradesh,1984,10,Narasapur,GEN,Vijaya Kumar Raju Bhupathiraju,M,Telugu Desam,TDP,366534,777048,1
588,Andhra Pradesh,1984,11,Eluru,GEN,Bolla Bulli Ramaiah,M,Telugu Desam,TDP,351340,790920,1


In [20]:
categorical_cols = [
    'st_name',
    'pc_name',
    'pc_type',
    'cand_name',
    'cand_sex',
    'partyname',
    'partyabbre'
]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

In [21]:
print(df.dtypes)

st_name       int64
year          int64
pc_no         int64
pc_name       int64
pc_type       int64
cand_name     int64
cand_sex      int64
partyname     int64
partyabbre    int64
totvotpoll    int64
electors      int64
winner        int64
dtype: object


In [22]:
X = df[
    [
        'st_name',
        'year',
        'pc_no',
        'pc_name',
        'pc_type',
        'cand_sex',
        'partyname',
        'partyabbre',
        'electors'
    ]
]

In [23]:
y = df['winner']

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)